# A receive-fixed swap: cashflows, value and risk

This notebook values a five-year USD swap in which we receive a fixed 4%
coupon on USD 1 million notional. We value it on its start date in an
illustrative SOFR market dated 24 February 2026. We then rebuild its present
value (PV) from the reported cashflows and look at how the PV responds to
each market quote.

The fixed leg pays semiannually and the floating leg pays quarterly. Dates are
not adjusted for holidays, and interest accrues on an Actual/360 basis. One
SOFR curve gives both the projected floating rates and the discount factors
for USD collateral. The `Swap` object holds the contract terms, and the
`Trade` object records that our position receives fixed.
[Setup](../../README.md) · [Par-rate calculation](par_swap.ipynb).


In [1]:
import pandas as pd
import quantsupport as qs
from market_data import load_market

market = load_market()
ref = market.quotes.reference_date
print("Market date:", ref)
market.quotes.to_dataframe().head(8)


Market date: 2026-02-24


,identifier,mid,bid,ask
0,CapletFloorlet_USD_SOFR_3M_10Y_Absolute_0.035_...,0.20,None,None
1,CapletFloorlet_USD_SOFR_3M_10Y_Absolute_0.045_...,0.18,None,None
2,CapletFloorlet_USD_SOFR_3M_10Y_Absolute_0.055_...,0.20,None,None
3,CapletFloorlet_USD_SOFR_3M_12Y_Absolute_0.035_...,0.19,None,None
4,CapletFloorlet_USD_SOFR_3M_12Y_Absolute_0.045_...,0.17,None,None
5,CapletFloorlet_USD_SOFR_3M_12Y_Absolute_0.055_...,0.19,None,None
6,CapletFloorlet_USD_SOFR_3M_1Y_Absolute_0.035_S...,0.33,None,None
7,CapletFloorlet_USD_SOFR_3M_1Y_Absolute_0.045_S...,0.29,None,None


## Dates and rate conventions

The contract needs dates, an accrual basis and payment frequencies:

- Adding a `Period` to a `Date` moves the date by that period, with no
  holiday adjustment.
- A `Calendar` adjusts a date to a business day when you ask it to.
- A `DayCounter` gives the year fraction used to accrue interest.

Conventions are enum values such as `qs.DayCounter.Actual360`. Rates are
decimals: `0.04` means 4%, and `0.0001` means one basis point.


In [2]:
six_months = qs.Period(6, qs.TimeUnit.Months)
calendar = qs.Calendar("UnitedStates(Sofr)")
end = ref + six_months
print("Unadjusted end:", end)
print("Adjusted end:", calendar.adjust(end, qs.BusinessDayConvention.ModifiedFollowing))
print("Actual/360 accrual:", qs.DayCounter.Actual360.year_fraction(ref, end))
print("Receive / pay signs:", qs.Side.LongReceive.sign(), qs.Side.PayShort.sign())


Unadjusted end: 2026-08-24
Adjusted end: 2026-08-24
Actual/360 accrual: 0.5027777777777778
Receive / pay signs: 1.0 -1.0


## The curve and its market index

`MarketIndex.SOFR` is the name that links the swap to the SOFR curve in the
market configuration. The curve is fitted to the deposit and OIS quotes listed
below. The fitted discount factors at those maturities are the points the
curve interpolates between.

The USD collateral (CSA) policy says to discount with SOFR in this example.
The same curve also projects the floating coupons. You can configure separate
curves for discounting and projection.


In [3]:
print(market.csa_policy)
market.curves[0].to_dict()


SingleCurveCSADiscountPolicy(collateral_currency='USD', collateral_index='SOFR')


{'day_counter': 'Actual360',
 'enable_extrapolation': True,
 'interpolator': 'LogLinear',
 'market_index': 'SOFR',
 'quotes': ['FixedRateDeposit_USD_SOFR_1D',
  'OIS_USD_SOFR_1Y',
  'OIS_USD_SOFR_2Y',
  'OIS_USD_SOFR_3Y',
  'OIS_USD_SOFR_5Y',
  'OIS_USD_SOFR_7Y',
  'OIS_USD_SOFR_10Y',
  'OIS_USD_SOFR_30Y']}

## Defining the position

We build the five-year swap and wrap it in a `Trade` with side
`LongReceive`, which means we receive fixed and pay floating. The swap's
identifier labels this position in the valuation and risk tables.


In [4]:
swap = qs.Swap(
    identifier="USD-RECEIVE-FIXED-5Y",
    start_date=ref,
    maturity_date=ref + "5Y",
    notional=1_000_000,
    fixed_rate=0.04,
    currency=qs.Currency.USD,
    market_index=qs.MarketIndex.SOFR,
    day_counter=qs.DayCounter.Actual360,
    fixed_leg_frequency=qs.Frequency.Semiannual,
    floating_leg_frequency=qs.Frequency.Quarterly,
)
trade = qs.Trade(swap, ref, qs.Side.LongReceive)
trade


Trade(instrument=Swap(compounding='Simple', currency='USD', day_counter='Actual360', fixed_leg_frequency='Semiannual', fixed_rate=0.04, floating_leg_frequency='Quarterly', identifier='USD-RECEIVE-FIXED-5Y', market_index='SOFR', maturity_date='2031-02-24', notional=1000000.0, rate_frequency='Semiannual', spread=None, start_date='2026-02-24'), trade_date='2026-02-24', side=LongReceive)

## Valuation in a pricing context

A `PricingContext` holds the market inputs and builds the curves the swap
needs. Inside the `with context:` block we ask for four results: the value,
the par rate, the cashflows and the sensitivities to market quotes. Curve
queries, such as a discount factor, must also run inside this block.

The numbers and pandas tables we keep stay usable after the block ends.


In [5]:
context = qs.PricingContext(
    quotes=market.quotes, curves=market.curves, csa_policy=market.csa_policy
)

with context:
    sofr = context.curve(qs.MarketIndex.SOFR)
    five_year_df = sofr.discount_factor(ref + "5Y")
    nodes = sofr.nodes()
    result = context.evaluate(
        trade,
        [
            qs.Request.Value,
            qs.Request.FairRate(0),
            qs.Request.Cashflows(),
            qs.Request.Sensitivities,
        ],
    )
    cashflows = result.cashflows
    risk = result.sensitivities

pd.Series(
    {
        "position_pv_usd": result.price,
        "par_fixed_rate": result.fair_rate,
        "five_year_discount_factor": five_year_df,
    }
)


position_pv_usd             -1664.991324
par_fixed_rate                  0.040368
five_year_discount_factor       0.817367
dtype: float64

## Interpreting the valuation

`price` is the PV of our position in USD, with its sign. `fair_rate` is the
fixed coupon that would give the swap zero value in this market. Our agreed
coupon of 4% is below the fair rate of about 4.037%. Receiving the lower
coupon gives the position a small negative value, about USD −1,665.

The cashflow table shows contract payments before discounting. Future
floating coupons use rates projected from today's curve. The section below
discounts these amounts to rebuild the PV.


In [6]:
cashflows[["payment_date", "leg_index", "type", "amount", "currency"]].head(10)


,payment_date,leg_index,type,amount,currency
0,2026-02-24,0,Disbursement,-1000000.000000,USD
1,2031-02-24,0,Redemption,1000000.000000,USD
2,2026-08-24,0,FixedRateCoupon,20111.111111,USD
3,2027-02-24,0,FixedRateCoupon,20444.444444,USD
4,2027-08-24,0,FixedRateCoupon,20111.111111,USD
5,2028-02-24,0,FixedRateCoupon,20444.444444,USD
6,2028-08-24,0,FixedRateCoupon,20222.222222,USD
7,2029-02-26,0,FixedRateCoupon,20666.666667,USD
8,2029-08-24,0,FixedRateCoupon,19888.888889,USD
9,2030-02-25,0,FixedRateCoupon,20555.555556,USD


The sensitivity table lists the market quotes the curve was fitted to. Each
`value` is the change in PV per unit change in that quote. The quotes are
decimal interest rates, so multiplying by `1e-4` gives the approximate USD
profit or loss, with its sign, if that quote rises by one basis point.


In [7]:
quote_risk = risk.assign(pnl_for_plus_1bp_usd=risk["value"] * 1e-4)
quote_risk.sort_values("pnl_for_plus_1bp_usd")


,pillar,value,pnl_for_plus_1bp_usd
6,OIS_USD_SOFR_5Y,-4.418095e+06,-441.809500
5,OIS_USD_SOFR_3Y,-6.761138e+03,-0.676114
2,OIS_USD_SOFR_1Y,-3.050580e+03,-0.305058
0,FixedRateDeposit_USD_SOFR_1D,-2.772806e+03,-0.277281
1,OIS_USD_SOFR_10Y,0.000000e+00,0.000000
4,OIS_USD_SOFR_30Y,0.000000e+00,0.000000
7,OIS_USD_SOFR_7Y,0.000000e+00,0.000000
3,OIS_USD_SOFR_2Y,1.333279e+03,0.133328


## Reconciling PV from cashflows

The swap value is the sum of discounted cashflows over legs $\ell$ and
payments $i$, where $s_\ell$ is the sign of each leg:
$V=\sum_{\ell,i}s_\ell C_{\ell i}P(0,T_{\ell i})$.

The cashflow table includes the signs:

- Leg 0 is the fixed leg and leg 1 is the floating leg.
- Amounts we receive are positive and amounts we pay are negative.
- An initial principal payment has the opposite sign to its repayment.

So we multiply each amount by its discount factor without applying the
trade direction again. The principal exchanges on the two legs cancel at the
start and at maturity.


In [8]:
reconciliation = cashflows.copy()
reconciliation["signed_amount_usd"] = reconciliation["amount"]
with context:
    curve = context.curve(qs.MarketIndex.SOFR)
    reconciliation["discount_factor"] = reconciliation["payment_date"].map(
        curve.discount_factor
    )
    opposite = context.evaluate(
        qs.Trade(swap, ref, qs.Side.PayShort), [qs.Request.Value]
    )
reconciliation["pv_usd"] = (
    reconciliation["signed_amount_usd"] * reconciliation["discount_factor"]
)
print(f"Receive-fixed PV: USD {result.price:,.2f}; par coupon: {result.fair_rate:.6%}.")
print(
    f"Cashflow reconstruction residual: USD {reconciliation['pv_usd'].sum() - result.price:.3e}."
)
print(
    "The 4% contract is above par."
    if result.fair_rate < 0.04
    else "The 4% contract is below par."
)
reconciliation.groupby("leg_index")[["signed_amount_usd", "pv_usd"]].sum()


Receive-fixed PV: USD -1,664.99; par coupon: 4.036802%.
Cashflow reconstruction residual: USD -1.637e-11.
The 4% contract is below par.


,signed_amount_usd,pv_usd
leg_index,,
0,202888.888889,-1.664991e+03
1,-202698.050026,-1.164153e-10


The leg totals show how much each leg contributes to the position value.
Reversing the trade side reverses the value, which checks the sign
convention. To see how the value depends on the agreed coupon in an unchanged
market, continue with the [par-rate calculation](par_swap.ipynb). For market
moves, see [quote shocks](../../volatilitysurface/curves_and_volatility.ipynb)
and [portfolio risk](../../sensitivity/portfolio_risk.ipynb).